In [15]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report
import time
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [16]:
#  1. Load & Inspect IMDB Dataset
dataset = load_dataset("imdb")

print(f"Train size : {len(dataset['train'])}")
print(f"Test  size : {len(dataset['test'])}")
print(f"\nExample review (truncated):\n  {dataset['train'][0]['text'][:200]}...")
print(f"  Label: {'positive' if dataset['train'][0]['label'] == 1 else 'negative'}")

Train size : 25000
Test  size : 25000

Example review (truncated):
  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
  Label: negative


In [25]:
#  2. Tokenizer
# bert-base-uncased: 12 layers, 768 hidden, 110M params
# Vocabulary: 30 522 WordPiece tokens (lowercase)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

sample_text = "The movie was absolutely fantastic!"
tokens = tokenizer.tokenize(sample_text)
print(f"\nTokenisation example:")
print(f"  Input  : {sample_text}")
print(f"  Tokens : {tokens}")
print(f"  IDs    : {tokenizer.convert_tokens_to_ids(tokens)}")


Tokenisation example:
  Input  : The movie was absolutely fantastic!
  Tokens : ['the', 'movie', 'was', 'absolutely', 'fantastic', '!']
  IDs    : [1996, 3185, 2001, 7078, 10392, 999]


In [26]:
# 3.Dataset Class
class IMDBDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=256):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        label = self.data[idx]['label']

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),  # [max_len]
            'attention_mask': encoding['attention_mask'].squeeze(0),  # [max_len]
            'token_type_ids': encoding['token_type_ids'].squeeze(0),  # [max_len]
            'label': torch.tensor(label, dtype=torch.long)
        }

MAX_LEN = 256
BATCH_SIZE = 16

train_dataset = IMDBDataset(dataset['train'].select(range(20000)), tokenizer, MAX_LEN)
test_data = dataset['test'].shuffle(seed=42).select(range(5000))
val_dataset = IMDBDataset(test_data, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
sample_batch = next(iter(train_loader))
print("\nSample batch shapes:")
print(f"  input_ids      : {sample_batch['input_ids'].shape}")
print(f"  attention_mask : {sample_batch['attention_mask'].shape}")
print(f"  labels         : {sample_batch['label'].shape}")




Sample batch shapes:
  input_ids      : torch.Size([16, 256])
  attention_mask : torch.Size([16, 256])
  labels         : torch.Size([16])


In [27]:
# 4.Model Architecture
class BertSentimentClassifier(nn.Module):
    """
    BERT + one linear classification head.
    Architecture:
        BERT encoder  →  [CLS] representation (768-d)
                      →  Dropout
                      →  Linear(768 → 2)
                      →  logits
    The [CLS] token's final hidden state is BERT's built-in
    sentence-level summary, so we use it directly for classification.
    """
    def __init__(self, bert_model_name='bert-base-uncased',
                 num_classes=2, dropout_rate=0.3):
        super().__init__()
        self.bert    = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc      = nn.Linear(self.bert.config.hidden_size, num_classes)
        # hidden_size = 768 for bert-base
    def forward(self, input_ids, attention_mask, token_type_ids):
        # outputs.last_hidden_state : [batch, seq_len, 768]
        # outputs.pooler_output     : [batch, 768]  (trained on [CLS])
        outputs = self.bert(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids
        )

        cls_output = outputs.last_hidden_state[:, 0, :]   # [batch, 768]

        out = self.dropout(cls_output)
        logits = self.fc(out)                              # [batch, 2]
        return logits


model = BertSentimentClassifier().to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"\nModel: BertSentimentClassifier")
print(f"Total trainable parameters: {count_parameters(model):,}")



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4688.01it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model: BertSentimentClassifier
Total trainable parameters: 109,483,778


In [28]:
# 5.Training Setup
N_EPOCHS   = 3
LR         = 2e-5
CLIP       = 1.0
WARMUP_PCT = 0.1        # 10 % of steps for LR warm-up
MODEL_PATH = 'bert_sentiment_best.pt'

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
    eps=1e-8
)
total_steps   = len(train_loader) * N_EPOCHS
warmup_steps  = int(total_steps * WARMUP_PCT)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

print(f"\nTraining setup:")
print(f"  Epochs       : {N_EPOCHS}")
print(f"  LR           : {LR}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Total steps  : {total_steps}")
print(f"  Warmup steps : {warmup_steps}")




Training setup:
  Epochs       : 3
  LR           : 2e-05
  Batch size   : 16
  Total steps  : 3750
  Warmup steps : 375


In [29]:
#  6. Train & Evaluate Functions
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask, token_type_ids)
        loss   = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels         = batch['label'].to(device)

            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels


def epoch_time(start, end):
    e = end - start
    return int(e / 60), int(e % 60)


In [30]:
# 7.Training Loop
best_val_loss = float('inf')
print("\nStarting fine-tuning...\n")

# Sanity check - 1 batch test
model.train()
batch = next(iter(train_loader))
input_ids      = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)
token_type_ids = batch['token_type_ids'].to(device)
labels         = batch['label'].to(device)

logits = model(input_ids, attention_mask, token_type_ids)
loss   = criterion(logits, labels)
loss.backward()

print(f"Test loss: {loss.item():.4f}")
print(f"Logits sample: {logits[:3]}")
print(f"Labels sample: {labels[:3]}")

for epoch in range(N_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

    mins, secs = epoch_time(t0, time.time())

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)

    print(f"Epoch {epoch+1:02}/{N_EPOCHS} | Time: {mins}m {secs}s")
    print(f"  Train  – Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"  Val    – Loss: {val_loss:.4f}  | Acc: {val_acc*100:.2f}%")
    print()

# Reload best checkpoint
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
print(f"Best model loaded from '{MODEL_PATH}'")




Starting fine-tuning...

Test loss: 0.6637
Logits sample: tensor([[-0.1330, -0.3177],
        [-0.5968, -0.1209],
        [-0.4576, -0.2015]], device='cuda:0', grad_fn=<SliceBackward0>)
Labels sample: tensor([1, 1, 0], device='cuda:0')
Epoch 01/3 | Time: 16m 21s
  Train  – Loss: 0.3057 | Acc: 86.94%
  Val    – Loss: 0.2242  | Acc: 91.70%

Epoch 02/3 | Time: 16m 7s
  Train  – Loss: 0.1591 | Acc: 94.80%
  Val    – Loss: 0.2827  | Acc: 91.88%

Epoch 03/3 | Time: 13m 1s
  Train  – Loss: 0.0758 | Acc: 98.06%
  Val    – Loss: 0.3952  | Acc: 91.38%

Best model loaded from 'bert_sentiment_best.pt'


In [31]:
#  8. Final Evaluation & Classification Report
_, final_acc, final_preds, final_labels = evaluate(model, val_loader, criterion)

print("=== FINAL EVALUATION RESULTS ===")
print(f"Accuracy: {final_acc*100:.2f}%\n")
print(classification_report(
    final_labels, final_preds,
    target_names=['negative', 'positive']
))

=== FINAL EVALUATION RESULTS ===
Accuracy: 91.70%

              precision    recall  f1-score   support

    negative       0.91      0.93      0.92      2494
    positive       0.93      0.90      0.92      2506

    accuracy                           0.92      5000
   macro avg       0.92      0.92      0.92      5000
weighted avg       0.92      0.92      0.92      5000



In [32]:
#  9. Inference – predict on new sentences
def predict_sentiment(texts, model, tokenizer, device, max_len=256):
    """
    texts : str or list of str
    Returns a list of dicts: [{'text': ..., 'label': ..., 'confidence': ...}]
    """
    if isinstance(texts, str):
        texts = [texts]

    model.eval()
    results = []

    for text in texts:
        enc = tokenizer(
            text,
            max_length     = max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )

        with torch.no_grad():
            logits = model(
                enc['input_ids'].to(device),
                enc['attention_mask'].to(device),
                enc['token_type_ids'].to(device)
            )

        probs      = torch.softmax(logits, dim=1).squeeze()
        label_idx  = probs.argmax().item()
        label_str  = 'positive' if label_idx == 1 else 'negative'
        confidence = probs[label_idx].item()

        results.append({
            'text'      : text[:80] + ('...' if len(text) > 80 else ''),
            'label'     : label_str,
            'confidence': confidence
        })

    return results


# Demo
demo_reviews = [
    "This film is an absolute masterpiece. The acting is superb and the story is deeply moving.",
    "Worst movie I have ever seen. Completely boring and a total waste of time.",
    "It was okay, nothing special but not terrible either.",
    "A stunning visual experience with a compelling narrative. Highly recommended!",
    "The plot made no sense and the characters were completely flat and uninteresting."
]

print("\n=== INFERENCE DEMO ===")
predictions = predict_sentiment(demo_reviews, model, tokenizer, device)
for p in predictions:
    bar = '█' * int(p['confidence'] * 20)
    print(f"\nText      : {p['text']}")
    print(f"Sentiment : {p['label'].upper():8s} | Confidence: {p['confidence']:.3f}  [{bar}]")




=== INFERENCE DEMO ===

Text      : This film is an absolute masterpiece. The acting is superb and the story is deep...
Sentiment : POSITIVE | Confidence: 0.997  [███████████████████]

Text      : Worst movie I have ever seen. Completely boring and a total waste of time.
Sentiment : NEGATIVE | Confidence: 0.999  [███████████████████]

Text      : It was okay, nothing special but not terrible either.
Sentiment : NEGATIVE | Confidence: 0.931  [██████████████████]

Text      : A stunning visual experience with a compelling narrative. Highly recommended!
Sentiment : POSITIVE | Confidence: 0.997  [███████████████████]

Text      : The plot made no sense and the characters were completely flat and uninteresting...
Sentiment : NEGATIVE | Confidence: 0.998  [███████████████████]
